# Hallucination Detector — Preprocessing
Run this notebook in Google Colab with GPU runtime.

In [1]:
# Step 1: Clone your GitHub repo into Colab
!git clone https://github.com/theek-sh-nahh/hallucination-detector.git
%cd hallucination-detector

Cloning into 'hallucination-detector'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 13 (delta 2), reused 13 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 4.04 KiB | 4.04 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/hallucination-detector


In [14]:
import sys

# Clear cached modules
mods = [k for k in sys.modules if 'src' in k or 'preprocess' in k]
for m in mods:
    del sys.modules[m]

# Pull latest code
!cd /content/hallucination-detector && git pull origin main

print("Done pulling.")

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.02 KiB | 349.00 KiB/s, done.
From https://github.com/theek-sh-nahh/hallucination-detector
 * branch            main       -> FETCH_HEAD
   bd6ce23..25811a1  main       -> origin/main
Updating bd6ce23..25811a1
Fast-forward
 src/preprocess.py | 81 ++++++++++++++++++++++++++++++++++++++++++++++++-------
 1 file changed, 72 insertions(+), 9 deletions(-)
Done pulling.


In [15]:
!pip install -q sentence-transformers
print("Dependencies installed.")

Dependencies installed.


In [16]:
!grep -n "HTTP\|parquet\|requests" /content/hallucination-detector/src/preprocess.py

69:#     Load TruthfulQA directly via HTTP parquet file.
72:#     import requests
75:#     print("Loading TruthfulQA dataset via HTTP...")
79:#         "/resolve/main/data/generation/validation-00000-of-00001.parquet"
82:#     response = requests.get(url, stream=True)
85:#     df_raw = pd.read_parquet(io.BytesIO(response.content))
117:    Load TruthfulQA by querying HuggingFace API for the real parquet URL,
120:    import requests
123:    print("Fetching TruthfulQA parquet URL from HuggingFace API...")
125:    # Step 1: Ask HuggingFace API for the real parquet file URL
126:    api_url = "https://datasets-server.huggingface.co/parquet?dataset=truthfulqa/truthful_qa"
127:    response = requests.get(api_url, timeout=30)
130:    parquet_files = response.json().get("parquet_files", [])
134:    for f in parquet_files:
140:        print("Available files:", [f['config'] + '/' + f['split'] for f in parquet_files])
141:        raise ValueError("Could not find generation/validation parquet file."

In [17]:
import sys
sys.path.insert(0, '/content/hallucination-detector')

from src.preprocess import run_pipeline

X_train, X_val, X_test, y_train, y_val, y_test, df = run_pipeline(
    raw_data_dir='data/raw',
    output_dir='data/processed'
)

HALLUCINATION DETECTOR — PREPROCESSING PIPELINE
Fetching TruthfulQA parquet URL from HuggingFace API...
  Found parquet at: https://huggingface.co/datasets/truthfulqa/truthful_qa/resolve/refs%2Fconvert%2Fparquet/generation/validation/0000.parquet
  Raw dataset shape: (817, 7)
  Loaded 5887 samples from TruthfulQA
  Custom samples file not found: data/raw/custom_samples.json — skipping

[2] Building DataFrame...
  DataFrame shape: (5882, 3)
  Class distribution:
label
factual           2587
hallucinated      1717
overconfident     1565
partially_true      13
Name: count, dtype: int64

[3] Generating embeddings...
  Loading SentenceTransformer: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Generating embeddings for 5882 texts...


Batches:   0%|          | 0/92 [00:00<?, ?it/s]

  Embeddings shape: (5882, 384)

[4] Splitting data...
  Train: (4116, 384), Val: (883, 384), Test: (883, 384)

[5] Saving processed data...
  All splits saved to: data/processed

Pipeline complete.


In [18]:
# Step 4: Verify the output
import numpy as np
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('Class counts:', {i: int((y_train==i).sum()) for i in range(4)})

X_train shape: (4116, 384)
y_train shape: (4116,)
Class counts: {0: 1811, 1: 1201, 2: 9, 3: 1095}


In [19]:
# Step 5: Zip and download processed data to your laptop
import shutil
shutil.make_archive('processed_data', 'zip', 'data/processed')

from google.colab import files
files.download('processed_data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>